### animação

In [6]:
import json

# Gerar arquivo HTML/JavaScript com todas as funcionalidades
def create_transit_simulation_html():
    # Dados das estrelas e coeficientes
    stars_data = {
        "M (0.2 R☉)": [0.2, 0.2, 3300],
        "K (0.7 R☉)": [0.7, 0.8, 4800],
        "G (1.0 R☉)": [1.0, 1.0, 5778],
        "F (1.3 R☉)": [1.3, 1.4, 6700],
        "A (1.7 R☉)": [1.7, 2.0, 8000]
    }
    
    ldc_data = {
        "M (0.2 R☉)": [0.5, 0.2],
        "K (0.7 R☉)": [0.4, 0.25],
        "G (1.0 R☉)": [0.3, 0.2],
        "F (1.3 R☉)": [0.2, 0.1],
        "A (1.7 R☉)": [0.1, 0.05]
    }
    
    html_content = f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Simulação de Trânsito Planetário</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            max-width: 1500px;
            margin: 0 auto;
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        .controls {{
            background: #f8f9fa;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 5px;
        }}
        .control-row {{
            display: flex;
            align-items: center;
            margin: 10px 0;
            gap: 20px;
        }}
        .control-group {{
            display: flex;
            align-items: center;
            gap: 10px;
        }}
        label {{
            font-weight: bold;
            min-width: 120px;
        }}
        input[type="range"] {{
            width: 200px;
        }}
        select {{
            padding: 5px;
            border-radius: 4px;
            border: 1px solid #ccc;
        }}
        button {{
            background: #007bff;
            color: white;
            border: none;
            padding: 10px 20px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 16px;
        }}
        button:hover {{
            background: #0056b3;
        }}
        .play-pause-btn {{
            background: #28a745;
            min-width: 80px;
        }}
        .play-pause-btn:hover {{
            background: #218838;
        }}
        .animation-controls {{
            background: #e9ecef;
            padding: 15px;
            border-radius: 8px;
            margin: 10px 0;
        }}
        .time-controls {{
            display: flex;
            align-items: center;
            gap: 15px;
            margin-top: 10px;
        }}
        #time-slider {{
            flex-grow: 1;
            min-width: 300px;
        }}
        .output-area {{
            display: flex;
            flex-direction: row;
            justify-content: center;
            align-items: flex-start;
            gap: 40px;
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 20px;
            background: white;
        }}
        .canvas-block {{
            display: flex;
            flex-direction: column;
            align-items: center;
            flex: 1 1 0;
            min-width: 450px;
            max-width: 700px;
        }}
        .canvas-title {{
            font-size: 15px;
            font-weight: bold;
            margin-bottom: 8px;
            text-align: center;
        }}
        #lightcurve {{
            width: 100%;
            height: 400px;
            border: 1px solid #ddd;
            margin-bottom: 0;
        }}
        #animation {{
            width: 100%;
            height: 400px;
            border: 1px solid #ddd;
            background: transparent;
            margin-bottom: 0;
        }}
        .value-display {{
            background: #e9ecef;
            padding: 2px 8px;
            border-radius: 3px;
            font-family: monospace;
        }}
        .time-display {{
            background: #d4edda;
            padding: 5px 10px;
            border-radius: 3px;
            font-family: monospace;
            min-width: 100px;
            text-align: center;
        }}
        .impact-value-display {{
            background: #e9ecef;
            padding: 2px 8px;
            border-radius: 3px;
            font-family: monospace;
            margin-left: 5px;
        }}
        .star-value-display {{
            background: #e9ecef;
            padding: 2px 8px;
            border-radius: 3px;
            font-family: monospace;
            margin-left: 1px;
            font-size: 14px;
            font-weight: normal;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1></h1>
        <div class="controls">
            <div class="control-row">
                <div class="control-group">
                    <label for="radius-slider">Raio do Planeta (R⊕):</label>
                    <input type="range" id="radius-slider" min="0.3" max="15" step="0.1" value="1.0">
                    <span class="value-display" id="radius-value">1.0</span>
                </div>
                <div class="control-group">
                    <label for="star-radius-slider">Raio da Estrela (R☉):</label> 
                    <input type="range" id="star-radius-slider" min="0.2" max="10" step="0.01" value="1.0">
                    <span class="star-value-display" id="star-radius-value">1.00</span>
                    <label for="star-radius-slider">Tipo Estelar: <span class="star-value-display" id="star-type-value">G</span></label> 
                </div>
            </div>
            <div class="control-row">
                <div class="control-group">
                    <label for="impact-slider">Parâmetro de impacto (b):</label>
                    <input type="range" id="impact-slider" min="-0.99" max="0.99" step="0.01" value="0">
                    <span class="impact-value-display" id="impact-value">0.00</span>
                </div>
                <div class="control-group">
                    <label>
                        <input type="checkbox" id="limb-darkening" checked>
                        Escurecimento de bordo
                    </label>
                </div>
                <!-- <button onclick="runSimulation()">Simular</button> -->
            </div>
        </div>
        
        <div class="animation-controls">
            <div class="control-row">
                <button class="play-pause-btn" id="play-pause-btn" onclick="togglePlayPause()">▶ Play</button>
                <button onclick="resetAnimation()">⏮ Reset</button>
                <span class="time-display" id="time-display">Tempo: 0.00 dias</span>
                <span class="value-display">Frame: <span id="frame-display">0</span>/300</span>
            </div>
            <div class="time-controls">
                <label for="time-slider">Linha do tempo:</label>
                <input type="range" id="time-slider" min="0" max="299" step="1" value="0">
                <div class="control-group">
                    <label for="speed-slider">Velocidade:</label>
                    <input type="range" id="speed-slider" min="10" max="200" step="10" value="40">
                    <span class="value-display" id="speed-value">40ms</span>
                </div>
            </div>
        </div>
        
        <div class="output-area">
            <div class="canvas-block">
                <div class="canvas-title">Curva de Luz</div>
                <canvas id="lightcurve"></canvas>
            </div>
            <div class="canvas-block">
                <div class="canvas-title">Interação entre Estrela e Planeta</div>
                <canvas id="animation"></canvas>
            </div>
        </div>
    </div>

    <div style="width:100%;text-align:center;margin-top:30px;">
        <span style="font-size:11px;color:#666;">
            Esse projeto é uma simulação de trânsito planetário, permitindo explorar diferentes parâmetros e suas influências nas curvas de luz. 
            Valores são representativos para fins educacionais levando em conta a física de trânsitos planetários e do escurecimento de limbo.
            </span>
    </div>

    <div style="width:100%;text-align:center;margin-top:5px;">
        <span style="font-size:11px;color:#666;">
            Desenvolvido por Ícaro Meidem - icarosilva@on.br<br>
            </span>
    </div>


    <script>
        // Dados das estrelas e coeficientes
        const stars = {json.dumps(stars_data)};
        const ldc_coeffs = {json.dumps(ldc_data)};
        
        // Variáveis globais para animação
        let animationData = null;
        let isPlaying = false;
        let currentFrame = 0;
        let animationInterval = null;
        let animationSpeed = 40;
        
        // Funções matemáticas
        function limb_darkening(mu, u1, u2) {{
            return 1 - u1 * (1 - mu) - u2 * Math.pow(1 - mu, 2);
        }}
        
        function earth_radius_to_rsun(rp_re) {{
            return rp_re * 0.0091577;
        }}
        
        function generate_transit_curve(z_array, rp_rs, u1, u2, use_ld) {{
            const flux = [];
            for (let i = 0; i < z_array.length; i++) {{
                const z = z_array[i];
                let delta = 0;
                
                if (z >= 1 + rp_rs) {{
                    delta = 0;
                }} else if (z <= 1 - rp_rs) {{
                    delta = Math.pow(rp_rs, 2);
                }} else {{
                    try {{
                        const k0 = Math.acos((Math.pow(z, 2) + Math.pow(rp_rs, 2) - 1) / (2 * z * rp_rs));
                        const k1 = Math.acos((Math.pow(z, 2) + 1 - Math.pow(rp_rs, 2)) / (2 * z));
                        const area = Math.pow(rp_rs, 2) * k0 + k1 - 0.5 * Math.sqrt(4 * Math.pow(z, 2) - Math.pow(1 + Math.pow(z, 2) - Math.pow(rp_rs, 2), 2));
                        delta = area / Math.PI;
                    }} catch (e) {{
                        delta = 0;
                    }}
                }}
                
                if (use_ld) {{
                    const mu = Math.sqrt(1 - Math.pow(Math.min(z, 1), 2));
                    delta *= limb_darkening(mu, u1, u2);
                }}
                
                flux.push((1 - delta) * 100);
            }}
            return flux;
        }}
        
        function linspace(start, stop, num) {{
            const arr = [];
            const step = (stop - start) / (num - 1);
            for (let i = 0; i < num; i++) {{
                arr.push(start + step * i);
            }}
            return arr;
        }}
        
        // Atualizar valores dos sliders
        document.getElementById('radius-slider').addEventListener('input', function() {{
            document.getElementById('radius-value').textContent = this.value;
        }});
        
        document.getElementById('speed-slider').addEventListener('input', function() {{
            animationSpeed = parseInt(this.value);
            document.getElementById('speed-value').textContent = this.value + 'ms';
            if (isPlaying) {{
                stopAnimation();
                startAnimation();
            }}
        }});
        
        document.getElementById('time-slider').addEventListener('input', function() {{
            currentFrame = parseInt(this.value);
            updateFrame();
        }});
        
        // Atualizar valor do parâmetro de impacto em tempo real e atualizar curva e animação
        document.getElementById('impact-slider').addEventListener('input', function() {{
            document.getElementById('impact-value').textContent = parseFloat(this.value).toFixed(2);
            // Atualizar curva de luz e animação em tempo real ao mover o slider
            if (animationData) {{
                runSimulation();
                // Redesenhar animação no frame atual após atualizar dados
                drawAnimationFrame(currentFrame);
            }}
        }});
        
        // Controles de animação
        function togglePlayPause() {{
            if (isPlaying) {{
                pauseAnimation();
            }} else {{
                playAnimation();
            }}
        }}
        
        function playAnimation() {{
            if (!animationData) return;
            isPlaying = true;
            document.getElementById('play-pause-btn').innerHTML = '⏸ Pause';
            startAnimation();
        }}
        
        function pauseAnimation() {{
            isPlaying = false;
            document.getElementById('play-pause-btn').innerHTML = '▶ Play';
            stopAnimation();
        }}
        
        function resetAnimation() {{
            pauseAnimation();
            currentFrame = 0;
            document.getElementById('time-slider').value = 0;
            updateFrame();
        }}
        
        function startAnimation() {{
            if (animationInterval) clearInterval(animationInterval);
            animationInterval = setInterval(() => {{
                currentFrame++;
                if (currentFrame >= animationData.x.length) {{
                    currentFrame = animationData.x.length - 1;
                    pauseAnimation(); // Parar no final
                    return;
                }}
                document.getElementById('time-slider').value = currentFrame;
                updateFrame();
            }}, animationSpeed);
        }}
        
        function stopAnimation() {{
            if (animationInterval) {{
                clearInterval(animationInterval);
                animationInterval = null;
            }}
        }}
        
        function updateFrame() {{
            if (!animationData) return;

            // Ajuste: tempo em dias começa em zero
            const t0 = animationData.time_days[0];
            const t = animationData.time_days[currentFrame] - t0;

            document.getElementById('frame-display').textContent = currentFrame;
            document.getElementById('time-display').textContent =
                `Tempo: ${{t.toFixed(2)}} dias`;

            drawAnimationFrame(currentFrame);
            drawLightCurveProgress(currentFrame);
        }}

        function runSimulation() {{
            const rp_re = parseFloat(document.getElementById('radius-slider').value);
            const rstar = parseFloat(document.getElementById('star-radius-slider').value);
            const starType = getStarType(rstar);
            
            // Estimar massa da estrela baseada no tipo espectral (para cálculos dinâmicos)
            const ms = (starType.mass_min + starType.mass_max) / 2;

            const use_ld = document.getElementById('limb-darkening').checked;
            const impact = parseFloat(document.getElementById('impact-slider').value);

            // Conversão do raio do planeta para raio solar
            const rp_rsun = rp_re * 0.0091577;
            const rp_rs = rp_rsun / rstar;

            // Gerar dados da órbita com parâmetro de impacto
            const x = linspace(-2, 2, 300);
            const z = x.map(val => Math.sqrt(val*val + impact*impact));
            const flux = generate_transit_curve(z, rp_rs, starType.u1, starType.u2, use_ld);

            const velocity = Math.sqrt(ms / 10);
            const time_days = x.map(val => val / velocity);

            animationData = {{
                x: x,
                flux: flux,
                time_days: time_days,
                rp_rs: rp_rs,
                star_type: starType.type,
                rp_re: rp_re,
                use_ld: use_ld,
                impact: impact,
                rstar: rstar,
                star_color: starType.color,
                star_data: starType
            }};

            document.getElementById('time-slider').max = x.length - 1;
            plotLightCurve(time_days, flux, starType.type, rp_re, use_ld);
            resetAnimation();
        }}

        function drawAnimationFrame(frameIndex) {{
            if (!animationData) return;
            const canvas = document.getElementById('animation');
            const ctx = canvas.getContext('2d');
            canvas.width = canvas.offsetWidth;
            canvas.height = canvas.offsetHeight;
            const centerX = canvas.width / 2;
            const centerY = canvas.height / 2;
            const starRadius = Math.min(canvas.height / 4, canvas.width / 10);
            const planetRadius = Math.max(starRadius * animationData.rp_rs, 4);

            ctx.clearRect(0, 0, canvas.width, canvas.height);

            // Desenhar estrela com cor conforme tipo espectral
            if (animationData.use_ld) {{
                // Desenhar estrela com escurecimento de bordo mais perceptível
                const steps = 50; // Menos steps para tornar mais visível
                for (let r = steps; r > 0; r--) {{
                    const frac = r / steps;
                    const mu = Math.sqrt(1 - frac * frac);
                    let intensity = 1 - animationData.star_data.u1 * (1 - mu) - animationData.star_data.u2 * Math.pow(1 - mu, 2);
                    
                    // Tornar o escurecimento mais suave
                    intensity = Math.max(0.4, Math.min(1, intensity)); // Mudou de 0.1 para 0.4 (escurecimento mais suave)
                    intensity = Math.pow(intensity, 0.9); // Mudou de 0.7 para 0.9 (menos contraste)
                    
                    ctx.beginPath();
                    ctx.arc(centerX, centerY, starRadius * frac, 0, 2 * Math.PI);
                    
                    // Criar gradiente radial para melhor visualização
                    const gradient = ctx.createRadialGradient(centerX, centerY, 0, centerX, centerY, starRadius * frac);
                    
                    // Converter cor base para RGB para manipular intensidade
                    const baseColor = animationData.star_color;
                    let r_val, g_val, b_val;
                    
                    if (baseColor.startsWith('#')) {{
                        r_val = parseInt(baseColor.slice(1, 3), 16);
                        g_val = parseInt(baseColor.slice(3, 5), 16);
                        b_val = parseInt(baseColor.slice(5, 7), 16);
                    }} else {{
                        // Fallback para cores nomeadas
                        r_val = 255; g_val = 220; b_val = 100;
                    }}
                    
                    // Aplicar intensidade à cor
                    const final_r = Math.round(r_val * intensity);
                    const final_g = Math.round(g_val * intensity);
                    const final_b = Math.round(b_val * intensity);
                    
                    gradient.addColorStop(0, `rgb(${{final_r}}, ${{final_g}}, ${{final_b}})`);
                    gradient.addColorStop(1, `rgb(${{Math.round(final_r * 0.8)}}, ${{Math.round(final_g * 0.8)}}, ${{Math.round(final_b * 0.8)}})`);
                    
                    ctx.fillStyle = gradient;
                    ctx.globalAlpha = 1.0;
                    ctx.fill();
                }}

            }} else {{

                //  Contorno da estrela uniforme
                ctx.beginPath();
                ctx.arc(centerX, centerY, starRadius, 0, 2 * Math.PI);
                ctx.strokeStyle = '#FFA500';
                ctx.lineWidth = 2;
                ctx.stroke();
            }}


            // Indicador de trânsito baseado no fluxo atual
            const currentFlux = animationData.flux[frameIndex];
            if (currentFlux < 99.999) {{
                ctx.strokeStyle = '#ff6b6b';
                ctx.lineWidth = 3;
                ctx.setLineDash([3, 3]);
                ctx.beginPath();
                ctx.arc(centerX, centerY, starRadius, 0, 2 * Math.PI);
                ctx.stroke();
                ctx.setLineDash([]);

                ctx.fillStyle = '#ff6b6b';
                ctx.font = 'bold 14px Arial';
                ctx.textAlign = 'center';
                ctx.fillText('TRÂNSITO DETECTADO', canvas.width / 2, canvas.height - 5);
            }}
        }}

        function drawLightCurveProgress(currentFrameIndex) {{
            if (!animationData) return;
            
            const canvas = document.getElementById('lightcurve');
            const ctx = canvas.getContext('2d');
            
            // Configurar canvas
            canvas.width = canvas.offsetWidth;
            canvas.height = canvas.offsetHeight;
            
            const margin = {{left: 60, right: 20, top: 50, bottom: 50}};
            const width = canvas.width - margin.left - margin.right;
            const height = canvas.height - margin.top - margin.bottom;
            
            // Limpar canvas
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            // Escalas
            const minTime = Math.min(...animationData.time_days);
            const maxTime = Math.max(...animationData.time_days);
            const minFlux = Math.min(...animationData.flux) - 0.05;
            const maxFlux = 100.05;
            
            // Função para converter coordenadas
            function scaleX(val) {{
                // Ajuste: tempo em dias começa em zero
                const t0 = animationData.time_days[0];
                const minTime = 0;
                const maxTime = animationData.time_days[animationData.time_days.length - 1] - t0;
                return margin.left + (val - t0) / (maxTime - minTime) * width;
            }}
            
            function scaleY(val) {{
                return margin.top + (maxFlux - val) / (maxFlux - minFlux) * height;
            }}
            
            // Desenhar eixos
            ctx.strokeStyle = '#000';
            ctx.lineWidth = 1;
            ctx.beginPath();
            ctx.moveTo(margin.left, margin.top);
            ctx.lineTo(margin.left, margin.top + height);
            ctx.lineTo(margin.left + width, margin.top + height);
            ctx.stroke();
            
            // Desenhar grid
            ctx.strokeStyle = '#ddd';
            ctx.lineWidth = 0.5;
            for (let i = 1; i < 10; i++) {{
                const y = margin.top + (height * i) / 10;
                ctx.beginPath();
                ctx.moveTo(margin.left, y);
                ctx.lineTo(margin.left + width, y);
                ctx.stroke();
            }}
            
            // Desenhar curva de luz até o frame atual (tempo real)
            if (currentFrameIndex >= 0) {{
                ctx.strokeStyle = '#000';
                ctx.lineWidth = 2;
                ctx.beginPath();
                let hasStarted = false;
                for (let i = 0; i <= currentFrameIndex; i++) {{
                    const x = scaleX(animationData.time_days[i]);
                    const y = scaleY(animationData.flux[i]);
                    if (!hasStarted) {{
                        ctx.moveTo(x, y);
                        hasStarted = true;
                    }} else {{
                        ctx.lineTo(x, y);
                    }}
                }}
                ctx.stroke();
            }}
            
            // Título e labels
            ctx.fillStyle = '#000';
            ctx.font = '14px Arial';
            ctx.textAlign = 'center';
            const title = `Curva de luz - Estrela do Tipo ${{animationData.star_type}} com ${{animationData.rp_re}} R⊕ ${{animationData.use_ld ? '(com LD)' : '(sem LD)'}}`;
            ctx.fillText(title, canvas.width / 2, 25);
            
            ctx.font = '14px Arial';
            ctx.fillText('Tempo (dias)', canvas.width / 2, canvas.height - 25);
            
            ctx.save();
            ctx.translate(15, canvas.height / 2);
            ctx.rotate(-Math.PI / 2);
            ctx.fillText('Fluxo relativo (%)', 0, 25);
            ctx.restore();
            
            // Mostrar fluxo atual apenas no display
            if (currentFrameIndex >= 0) {{
                const currentFlux = animationData.flux[currentFrameIndex];
                ctx.fillStyle = '#000';
                ctx.font = '12px Arial';
                ctx.textAlign = 'left';
                ctx.fillText(`Fluxo atual: ${{currentFlux.toFixed(3)}}%`, margin.left + 10, margin.top + 190);

                // Se for o último frame, mostrar fluxo mínimo registrado e profundidade máxima observada
                if (currentFrameIndex === animationData.flux.length - 1) {{
                    const minFlux = Math.min(...animationData.flux);
                    const profundidadeMaxima = 100 - minFlux;
                    ctx.fillText(`Fluxo mínimo registrado: ${{minFlux.toFixed(3)}}% (máximo bloqueio pelo planeta)`, margin.left + 10, margin.top + 210);
                    ctx.fillText(`Profundidade máxima observada: ${{profundidadeMaxima.toFixed(3)}}% do fluxo total emitido pela estrela`, margin.left + 10, margin.top + 230);
                }}
            }}
        }}
        
        function plotLightCurve(time_days, flux, star_type, rp_re, use_ld) {{
            // Inicializar com canvas vazio (apenas eixos)
            drawLightCurveProgress(-1);
        }}
        
        // Executar simulação inicial
        window.onload = function() {{
            runSimulation();
        }};

        // Tabela de tipos espectrais por raio solar
        const starTypes = [
            {{ type: "O", color: "#6bb6ff", min: 6.6, max: 10, mass_min: 16, mass_max: 50, temp: 35000, u1: 0.15, u2: 0.05 }},
            {{ type: "B", color: "#a3c6ff", min: 1.8, max: 6.6, mass_min: 2.1, mass_max: 16, temp: 20000, u1: 0.18, u2: 0.07 }},
            {{ type: "A", color: "#e6eaff", min: 1.4, max: 1.8, mass_min: 1.4, mass_max: 2.1, temp: 9000, u1: 0.22, u2: 0.09 }},
            {{ type: "F", color: "#fffbe7", min: 1.15, max: 1.4, mass_min: 1.04, mass_max: 1.4, temp: 7000, u1: 0.25, u2: 0.12 }},
            {{ type: "G", color: "#fff5b1", min: 0.96, max: 1.15, mass_min: 0.8, mass_max: 1.04, temp: 5800, u1: 0.3, u2: 0.2 }},
            {{ type: "K", color: "#ffd27a", min: 0.7, max: 0.96, mass_min: 0.45, mass_max: 0.8, temp: 4500, u1: 0.4, u2: 0.25 }},
            {{ type: "M", color: "#ffb07a", min: 0.2, max: 0.7, mass_min: 0.08, mass_max: 0.45, temp: 3200, u1: 0.5, u2: 0.2 }}
        ];

        // Função para determinar tipo espectral com base no raio
        function getStarType(radius) {{
            for (let i = 0; i < starTypes.length; i++) {{
                if (radius >= starTypes[i].min && (radius < starTypes[i].max || i === 0)) {{
                    return starTypes[i];
                }}
            }}
            return starTypes[starTypes.length - 1]; // M como padrão
        }}

        // Atualizar valor do raio da estrela e tipo espectral em tempo real
        document.getElementById('star-radius-slider').addEventListener('input', function() {{
            const rstar = parseFloat(this.value);
            document.getElementById('star-radius-value').textContent = rstar.toFixed(2);
            const starType = getStarType(rstar);
            document.getElementById('star-type-value').textContent = starType.type;
            if (animationData) {{
                runSimulation();
                drawAnimationFrame(currentFrame);
            }}
        }});

        // Atualizar valor do raio do planeta em tempo real e atualizar curva e animação
        document.getElementById('radius-slider').addEventListener('input', function() {{
            document.getElementById('radius-value').textContent = this.value;
            if (animationData) {{
                runSimulation();
                drawAnimationFrame(currentFrame);
            }}
        }});
        
        document.getElementById('speed-slider').addEventListener('input', function() {{
            animationSpeed = parseInt(this.value);
            document.getElementById('speed-value').textContent = this.value + 'ms';
            if (isPlaying) {{
                stopAnimation();
                startAnimation();
            }}
        }});
        
        document.getElementById('time-slider').addEventListener('input', function() {{
            currentFrame = parseInt(this.value);
            updateFrame();
        }});
        
        // Atualizar valor do parâmetro de impacto em tempo real e atualizar curva e animação
        document.getElementById('impact-slider').addEventListener('input', function() {{
            document.getElementById('impact-value').textContent = parseFloat(this.value).toFixed(2);
            // Atualizar curva de luz e animação em tempo real ao mover o slider
            if (animationData) {{
                runSimulation();
                // Redesenhar animação no frame atual após atualizar dados
                drawAnimationFrame(currentFrame);
            }}
        }});
        
        // Controles de animação
        function togglePlayPause() {{
            if (isPlaying) {{
                pauseAnimation();
            }} else {{
                playAnimation();
            }}
        }}
        
        function playAnimation() {{
            if (!animationData) return;
            isPlaying = true;
            document.getElementById('play-pause-btn').innerHTML = '⏸ Pause';
            startAnimation();
        }}
        
        function pauseAnimation() {{
            isPlaying = false;
            document.getElementById('play-pause-btn').innerHTML = '▶ Play';
            stopAnimation();
        }}
        
        function resetAnimation() {{
            pauseAnimation();
            currentFrame = 0;
            document.getElementById('time-slider').value = 0;
            updateFrame();
        }}
        
        function startAnimation() {{
            if (animationInterval) clearInterval(animationInterval);
            animationInterval = setInterval(() => {{
                currentFrame++;
                if (currentFrame >= animationData.x.length) {{
                    currentFrame = animationData.x.length - 1;
                    pauseAnimation(); // Parar no final
                    return;
                }}
                document.getElementById('time-slider').value = currentFrame;
                updateFrame();
            }}, animationSpeed);
        }}
        
        function stopAnimation() {{
            if (animationInterval) {{
                clearInterval(animationInterval);
                animationInterval = null;
            }}
        }}
        
        function updateFrame() {{
            if (!animationData) return;

            // Ajuste: tempo em dias começa em zero
            const t0 = animationData.time_days[0];
            const t = animationData.time_days[currentFrame] - t0;

            document.getElementById('frame-display').textContent = currentFrame;
            document.getElementById('time-display').textContent =
                `Tempo: ${{t.toFixed(2)}} dias`;

            drawAnimationFrame(currentFrame);
            drawLightCurveProgress(currentFrame);
        }}

        function runSimulation() {{
            const rp_re = parseFloat(document.getElementById('radius-slider').value);
            const rstar = parseFloat(document.getElementById('star-radius-slider').value);
            const starType = getStarType(rstar);
            
            // Estimar massa da estrela baseada no tipo espectral (para cálulos dinâmicos)
            const ms = (starType.mass_min + starType.mass_max) / 2;

            const use_ld = document.getElementById('limb-darkening').checked;
            const impact = parseFloat(document.getElementById('impact-slider').value);

            // Conversão do raio do planeta para raio solar
            const rp_rsun = rp_re * 0.0091577;
            const rp_rs = rp_rsun / rstar;

            // Gerar dados da órbita com parâmetro de impacto
            const x = linspace(-2, 2, 300);
            const z = x.map(val => Math.sqrt(val*val + impact*impact));
            const flux = generate_transit_curve(z, rp_rs, starType.u1, starType.u2, use_ld);

            const velocity = Math.sqrt(ms / 10);
            const time_days = x.map(val => val / velocity);

            animationData = {{
                x: x,
                flux: flux,
                time_days: time_days,
                rp_rs: rp_rs,
                star_type: starType.type,
                rp_re: rp_re,
                use_ld: use_ld,
                impact: impact,
                rstar: rstar,
                star_color: starType.color,
                star_data: starType
            }};

            document.getElementById('time-slider').max = x.length - 1;
            plotLightCurve(time_days, flux, starType.type, rp_re, use_ld);
            resetAnimation();
        }}

        function drawAnimationFrame(frameIndex) {{
            if (!animationData) return;
            const canvas = document.getElementById('animation');
            const ctx = canvas.getContext('2d');
            canvas.width = canvas.offsetWidth;
            canvas.height = canvas.offsetHeight;
            const centerX = canvas.width / 2;
            const centerY = canvas.height / 2;
            const starRadius = Math.min(canvas.height / 3, canvas.width / 5);
            const planetRadius = Math.max(starRadius * animationData.rp_rs, 3);

            ctx.clearRect(0, 0, canvas.width, canvas.height);

            // Desenhar estrela com cor conforme tipo espectral
            if (animationData.use_ld) {{
                // Desenhar estrela com escurecimento de bordo mais perceptível
                const steps = 50; // Menos steps para tornar mais visível
                for (let r = steps; r > 0; r--) {{
                    const frac = r / steps;
                    const mu = Math.sqrt(1 - frac * frac);
                    let intensity = 1 - animationData.star_data.u1 * (1 - mu) - animationData.star_data.u2 * Math.pow(1 - mu, 2);
                    
                    // Tornar o escurecimento mais suave
                    intensity = Math.max(0.4, Math.min(1, intensity)); // Mudou de 0.1 para 0.4 (escurecimento mais suave)
                    intensity = Math.pow(intensity, 0.9); // Mudou de 0.7 para 0.9 (menos contraste)
                    
                    ctx.beginPath();
                    ctx.arc(centerX, centerY, starRadius * frac, 0, 2 * Math.PI);
                    
                    // Criar gradiente radial para melhor visualização
                    const gradient = ctx.createRadialGradient(centerX, centerY, 0, centerX, centerY, starRadius * frac);
                    
                    // Converter cor base para RGB para manipular intensidade
                    const baseColor = animationData.star_color;
                    let r_val, g_val, b_val;
                    
                    if (baseColor.startsWith('#')) {{
                        r_val = parseInt(baseColor.slice(1, 3), 16);
                        g_val = parseInt(baseColor.slice(3, 5), 16);
                        b_val = parseInt(baseColor.slice(5, 7), 16);
                    }} else {{
                        // Fallback para cores nomeadas
                        r_val = 255; g_val = 220; b_val = 100;
                    }}
                    
                    // Aplicar intensidade à cor
                    const final_r = Math.round(r_val * intensity);
                    const final_g = Math.round(g_val * intensity);
                    const final_b = Math.round(b_val * intensity);
                    
                    gradient.addColorStop(0, `rgb(${{final_r}}, ${{final_g}}, ${{final_b}})`);
                    gradient.addColorStop(1, `rgb(${{Math.round(final_r * 0.8)}}, ${{Math.round(final_g * 0.8)}}, ${{Math.round(final_b * 0.8)}})`);
                    
                    ctx.fillStyle = gradient;
                    ctx.globalAlpha = 1.0;
                    ctx.fill();
                }}
                
                // Adicionar borda escura para destacar o escurecimento
                ctx.beginPath();
                ctx.arc(centerX, centerY, starRadius, 0, 2 * Math.PI);
                ctx.strokeStyle = `rgba(0.1, 0.2, 0.5, 0.15)`; // Mudou de 0.3 para 0.15 (borda mais suave)
                ctx.lineWidth = 1; // Mudou de 3 para 2 (linha mais fina)
                ctx.stroke();
            }} else {{
                // Desenhar estrela uniforme
                ctx.beginPath();
                ctx.arc(centerX, centerY, starRadius, 0, 2 * Math.PI);
                ctx.fillStyle = animationData.star_color;
                ctx.globalAlpha = 2.0;
                ctx.fill();
                
            }}


            // Desenhar órbita (linha pontilhada)
            ctx.setLineDash([5, 5]);
            ctx.strokeStyle = '#666666';
            ctx.lineWidth = 1;
            ctx.beginPath();
            ctx.moveTo(centerX - starRadius * 2, centerY + animationData.impact * starRadius);
            ctx.lineTo(centerX + starRadius * 2, centerY + animationData.impact * starRadius);
            ctx.stroke();
            ctx.setLineDash([]);

            // Desenhar planeta
            const x_position = animationData.x[frameIndex];
            const planetX = centerX + x_position * starRadius;
            const planetY = centerY + animationData.impact * starRadius;
            ctx.beginPath();
            ctx.arc(planetX, planetY, planetRadius, 0, 2 * Math.PI);
            ctx.fillStyle = 'black';
            ctx.fill();
            ctx.strokeStyle = '#333';
            ctx.lineWidth = 1;
            ctx.stroke();

            // Indicador de trânsito baseado no fluxo atual
            const currentFlux = animationData.flux[frameIndex];
            if (currentFlux < 99.999) {{
                ctx.strokeStyle = '#ff6b6b';
                ctx.lineWidth = 3;
                ctx.setLineDash([3, 3]);
                ctx.beginPath();
                ctx.arc(centerX, centerY, starRadius, 0, 2 * Math.PI);
                ctx.stroke();
                ctx.setLineDash([]);

                ctx.fillStyle = '#ff6b6b';
                ctx.font = 'bold 14px Arial';
                ctx.textAlign = 'center';
                ctx.fillText('TRÂNSITO DETECTADO', canvas.width / 2, canvas.height -  5);
            }}
        }}

        function drawLightCurveProgress(currentFrameIndex) {{
            if (!animationData) return;
            
            const canvas = document.getElementById('lightcurve');
            const ctx = canvas.getContext('2d');
            
            // Configurar canvas
            canvas.width = canvas.offsetWidth;
            canvas.height = canvas.offsetHeight;
            
            const margin = {{left: 60, right: 20, top: 50, bottom: 50}};
            const width = canvas.width - margin.left - margin.right;
            const height = canvas.height - margin.top - margin.bottom;
            
            // Limpar canvas
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            // Escalas
            const minTime = Math.min(...animationData.time_days);
            const maxTime = Math.max(...animationData.time_days);
            const minFlux = Math.min(...animationData.flux) - 0.05;
            const maxFlux = 100.05;
            
            // Função para converter coordenadas
            function scaleX(val) {{
                // Ajuste: tempo em dias começa em zero
                const t0 = animationData.time_days[0];
                const minTime = 0;
                const maxTime = animationData.time_days[animationData.time_days.length - 1] - t0;
                return margin.left + (val - t0) / (maxTime - minTime) * width;
            }}
            
            function scaleY(val) {{
                return margin.top + (maxFlux - val) / (maxFlux - minFlux) * height;
            }}
            
            // Desenhar eixos
            ctx.strokeStyle = '#000';
            ctx.lineWidth = 1;
            ctx.beginPath();
            ctx.moveTo(margin.left, margin.top);
            ctx.lineTo(margin.left, margin.top + height);
            ctx.lineTo(margin.left + width, margin.top + height);
            ctx.stroke();
            
            // Desenhar grid
            ctx.strokeStyle = '#ddd';
            ctx.lineWidth = 0.5;
            for (let i = 1; i < 10; i++) {{
                const y = margin.top + (height * i) / 10;
                ctx.beginPath();
                ctx.moveTo(margin.left, y);
                ctx.lineTo(margin.left + width, y);
                ctx.stroke();
            }}
            
            // Desenhar curva de luz até o frame atual (tempo real)
            if (currentFrameIndex >= 0) {{
                ctx.strokeStyle = '#000';
                ctx.lineWidth = 2;
                ctx.beginPath();
                let hasStarted = false;
                for (let i = 0; i <= currentFrameIndex; i++) {{
                    const x = scaleX(animationData.time_days[i]);
                    const y = scaleY(animationData.flux[i]);
                    if (!hasStarted) {{
                        ctx.moveTo(x, y);
                        hasStarted = true;
                    }} else {{
                        ctx.lineTo(x, y);
                    }}
                }}
                ctx.stroke();
            }}
            
            // Título e labels
            ctx.fillStyle = '#000';
            ctx.font = '14px Arial';
            ctx.textAlign = 'center';
            const title = `Curva de luz - Estrela do Tipo ${{animationData.star_type}} com ${{animationData.rp_re}} R⊕ ${{animationData.use_ld ? '(com LD)' : '(sem LD)'}}`;
            ctx.fillText(title, canvas.width / 2, 25);
            
            ctx.font = '14px Arial';
            ctx.fillText('Tempo (dias)', canvas.width / 1.1, canvas.height - 35);
            
            ctx.save();
            ctx.translate(15, canvas.height / 2);
            ctx.rotate(-Math.PI / 2);
            ctx.fillText('Fluxo relativo (%)', 0, 38);
            ctx.restore();
            
            // Mostrar fluxo atual apenas no display
            if (currentFrameIndex >= 0) {{
                const currentFlux = animationData.flux[currentFrameIndex];
                ctx.fillStyle = '#000';
                ctx.font = '12px Arial';
                ctx.textAlign = 'left';
                ctx.fillText(`Fluxo atual: ${{currentFlux.toFixed(3)}}%`, margin.left + 10, margin.top + 290);

                // Se for o último frame, mostrar fluxo mínimo registrado e profundidade máxima observada
                if (currentFrameIndex === animationData.flux.length - 1) {{
                    const minFlux = Math.min(...animationData.flux);
                    const profundidadeMaxima = 100 - minFlux;
                    // ctx.fillText(`Fluxo mínimo registrado: ${{minFlux.toFixed(3)}}% (máximo bloqueio pelo planeta)`, margin.left + 10, margin.top + 210);
                    ctx.fillText(`O planeta bloqueou ${{profundidadeMaxima.toFixed(3)}}% do fluxo total da estrela`, margin.left + 10, margin.top + 320);
                }}
            }}
        }}
        
        function plotLightCurve(time_days, flux, star_type, rp_re, use_ld) {{
            // Inicializar com canvas vazio (apenas eixos)
            drawLightCurveProgress(-1);
        }}
        
        // Executar simulação inicial
        window.onload = function() {{
            runSimulation();
        }};

        // Atualizar animação ao mudar o checkbox de escurecimento de bordo
        document.getElementById('limb-darkening').addEventListener('change', function() {{
            if (animationData) {{
                runSimulation();
                drawAnimationFrame(currentFrame);
            }}
        }});
    </script>
</body>
</html>'''
    
    # Salvar arquivo HTML
    with open('/home/usuario/ON/ON-backup/doutorado/artigo_snea/transit_simulation.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print("Arquivo 'transit_simulation.html' criado com sucesso!")
    print("Abra o arquivo no navegador para usar a simulação.")

# Executar a criação do arquivo
create_transit_simulation_html()

Arquivo 'transit_simulation.html' criado com sucesso!
Abra o arquivo no navegador para usar a simulação.
